# 04 — MCP Tools (stdio transport)

Shows how to:
1. Connect to an **MCP server** launched as a subprocess (stdio transport)
2. Auto-discover all tools the server exposes
3. Generate an LLM response that uses those tools

This example uses the public `@modelcontextprotocol/server-filesystem` package.

**Prerequisites**: `node` / `npx` available on `PATH` and `OPENAI_API_KEY` set.

In [2]:

from raavan.extensions.mcp import MCPClient, MCPTool
from raavan.providers.llm.openai.openai_client import OpenAIClient
from raavan.core.memory.unbounded_memory import UnboundedMemory
from raavan.core.messages.client_messages import UserMessage, SystemMessage

## Connect and discover tools

In [ ]:
async def demo():
    mcp_client = MCPClient()
    try:
        await mcp_client.connect(
            command='npx',
            args=['-y', '@modelcontextprotocol/server-filesystem', '/tmp'],
        )
        print('Connected to MCP server')

        mcp_tools = await MCPTool.from_mcp_client(mcp_client)
        print(f'Discovered {len(mcp_tools)} tools:')
        for t in mcp_tools:
            print(f'  - {t.name}: {t.description}')

        client = OpenAIClient(model='gpt-4o')
        memory = UnboundedMemory()
        memory.add_message(SystemMessage(content='You are a helpful assistant with filesystem access.'))
        memory.add_message(UserMessage(content=[{'type': 'text', 'text': 'List the files in the /tmp directory'}]))

        # get_openai_schema() → OpenAI function-calling dict (not get_schema() → internal Tool object)
        response = await client.generate(
            messages=memory.get_messages(),
            tools=[t.get_openai_schema() for t in mcp_tools],
        )
        print(f'\nAgent response: {response.content}')
        if response.tool_calls:
            print('Tool calls requested:')
            for tc in response.tool_calls:
                # ToolCallMessage: use tc.name and tc.arguments, not tc.function['name']
                print(f'  - {tc.name}({tc.arguments})')

    except Exception as e:
        print(f'⚠ Error: {e}')
        print('  Requires: Node.js + npx  (npm install -g npx)')
    finally:
        if mcp_client.is_connected:
            await mcp_client.disconnect()
            print('Disconnected')

await demo()


⚠ Error: Failed to connect to MCP server via stdio: fileno
  Requires: Node.js + npx  (npm install -g npx)
